### Preparings for Part 2: Exploratory Case Study on the Penmanshiel Dataset

In [ ]:
from helperfunctions import helper as hfn
from helperfunctions import intern_constants as ic
from helperfunctions.pretty_print import PrettyPrint as pp
from helperfunctions import training_lib as tl
from helperfunctions.preprocessing import PreprocessingStep5
from helperfunctions.detection import Part2 as p2
from torch import nn
import pandas as pd
from pathlib import Path
import os
from glob import glob

pd.set_option("display.max_columns", None)

In [ ]:
cfg_part2 = hfn.TrainConfig(config_name="part2", choose_val_set=2)

In [ ]:
print(f"test start:{cfg_part2.test_start_time}\n"
      f"test end:  {cfg_part2.test_end_time}")

In [ ]:
_, _, test_loader = hfn.build_dataloaders(
    train_csv_dir=ic.PATH_PC_FILTERING,
    val_csv_dir=ic.PATH_IMPUTED,
    test_csv_dir=ic.PATH_IMPUTED,
    cfg=cfg_part2
)

### Load Autoencoder

In [ ]:
best = tl.get_model_results(ic.PATH_TO_BEST_MODEL_DIR, best_n=1)
ae, _, _, _, _ = best[0]

ae = ae.to(cfg_part2.device).eval()

loss_fn = nn.MSELoss(reduction="none")


### Load threshold 

In [ ]:
theta = p2.load_threshold_table()

In [ ]:
test_eval_df = tl.eval_model(
    model=ae,
    data_loader=test_loader,
    device=cfg_part2.device,
    loss_fn = loss_fn
)
display(test_eval_df.head())

os.makedirs(ic.PATH_PART2_EVAL_TEST, exist_ok=True)
test_eval_df.to_csv(ic.PATH_PART2_EVAL_TEST / "eval_test_set.csv", index=False)


### We need to drop the Imputations from test_eval_df

In [ ]:
test_eval_clean = p2.drop_imputations(test_eval_df)

In [ ]:
detections_df = test_eval_clean[test_eval_clean[ic.MEAN_LOSS_PER_SAMPLE] >= theta]
detections_df = detections_df.sort_values(by=ic.MEAN_LOSS_PER_SAMPLE, ascending=False).reset_index(drop=True)

In [ ]:
display(detections_df.head())
display(len(detections_df))

In [ ]:
selected_detections = p2.select_top_detections_with_gap(
    det_df=detections_df,
    top_n=None,
    gap_days=1,
    save_filename="part2_dets_gdays_1.csv"
)

In [ ]:
display(selected_detections)

### Power Curve 

In [ ]:
pre_step5 = PreprocessingStep5()
df_pc = pre_step5._prepare_power_curve()
df_pc.head()

In [ ]:
test_start = cfg_part2.test_start_time
test_end = cfg_part2.test_end_time

wind_col = "Wind speed (m/s)"
power_col = "Power (kW)"

imputed_files = sorted(Path(ic.PATH_IMPUTED).glob("*.csv"))

wt_farm : list[pd.DataFrame] = []

for file in imputed_files:
    df_wt = pd.read_csv(file, parse_dates=[ic.TS_COL])
    df_wt[ic.TS_COL]= pd.to_datetime(df_wt[ic.TS_COL])
    df_wt = df_wt.set_index(ic.TS_COL).sort_index()
    
    df_wt_test = df_wt.loc[test_start:test_end].copy()
    
    df_wt_test[ic.TS_COL] = df_wt_test.index
    wt_farm.append(df_wt_test.reset_index(drop=True))
    
wt_farm_df = pd.concat(wt_farm, axis=0)
wt_farm_df = wt_farm_df.sort_values([ic.WT_ID, ic.TS_COL])
wt_farm_df.head()

wt_farm_pc_df = wt_farm_df[[c for c in [ic.WT_ID, ic.TS_COL, wind_col, power_col] if c in wt_farm_df.columns]].copy()

wt_farm_path = ic.PATH_PART2_WT_FARM
os.makedirs(wt_farm_path, exist_ok=True)

wt_farm_pc_df.to_csv(wt_farm_path /  "wt_farm_pc_df.csv", index=False)

In [ ]:
fps = glob(os.path.join(ic.PATH_PART2_DETECTIONS, "part2_dets_gdays_1.csv"))
print(fps)
fp = fps[0]
df_detections = pd.read_csv(fp, parse_dates=[ic.TS_COL])
df_detections[ic.TS_COL] = pd.to_datetime(df_detections[ic.TS_COL])


df_det_keys = df_detections[[ic.WT_ID, ic.TS_COL]].copy()

df_det_points = df_det_keys.merge(
    wt_farm_pc_df[[ic.WT_ID,ic.TS_COL, wind_col, power_col]],
    on=[ic.WT_ID, ic.TS_COL],
    how="left",
    validate="one_to_one"
)

df_det_points.head()

In [ ]:
pp.print_powercurve(df_pc=df_pc,
                    df_wts=p2.drop_imputations( wt_farm_pc_df),
                    df_detections=df_det_points,
                    detections_label="All filtered Detections"
                    )

In [ ]:
sel_det_no_gaps = p2.select_top_detections_with_gap(
    det_df=detections_df,
    top_n=None,
    gap_days=0,
    save_filename="part2_dets_no_gaps.csv"
)

df_det_keys_ng = sel_det_no_gaps[[ic.WT_ID, ic.TS_COL]].copy()

df_det_points_ng = df_det_keys_ng.merge(
    wt_farm_pc_df[[ic.WT_ID,ic.TS_COL, wind_col, power_col]],
    on=[ic.WT_ID, ic.TS_COL],
    how="left",
    validate="one_to_one"
)

df_det_points_ng.head()

df_det_points_ng.to_csv(ic.PATH_PART2_DETECTIONS/ "det_points_no_gap.csv", index=False)

pp.print_powercurve(df_pc=df_pc,
                    df_wts=p2.drop_imputations( wt_farm_pc_df),
                    df_detections=df_det_points_ng,
                    detections_label="All Detections"
                    )



In [ ]:
print(f"total samples (no imputations): {len(p2.drop_imputations( wt_farm_pc_df))}")
print(f"Total Number of detections: {len(df_det_points_ng)}")
print(f"Share of detections: {len(df_det_points_ng) / len(p2.drop_imputations( wt_farm_pc_df))}")